In [0]:
CREATE SCHEMA IF NOT EXISTS marketing_analytics.gold;

CREATE OR REPLACE TABLE marketing_analytics.gold.ab_test_results AS
SELECT
  test_group,
  COUNT(*)                                          AS users,
  COUNT_IF(converted)                               AS conversions,
  ROUND(100.0 * COUNT_IF(converted) / COUNT(*), 3)  AS conversion_rate_pct
FROM marketing_analytics.silver.ab_test_clean
GROUP BY test_group;

SELECT * FROM marketing_analytics.gold.ab_test_results;

In [0]:
SELECT
  test_group,
  COUNT(*)                     AS users,
  SUM(total_ads)               AS total_impressions,
  ROUND(AVG(total_ads), 1)     AS avg_ads_per_user,
  PERCENTILE(total_ads, 0.5)   AS median_ads_per_user
FROM marketing_analytics.silver.ab_test_clean
GROUP BY test_group;

In [0]:
CREATE OR REPLACE TABLE marketing_analytics.gold.ab_test_by_frequency AS
SELECT
  CASE
    WHEN total_ads <= 5   THEN '1) 1-5'
    WHEN total_ads <= 20  THEN '2) 6-20'
    WHEN total_ads <= 50  THEN '3) 21-50'
    WHEN total_ads <= 100 THEN '4) 51-100'
    WHEN total_ads <= 500 THEN '5) 101-500'
    ELSE                       '6) 500+'
  END                                               AS ad_bucket,
  test_group,
  COUNT(*)                                          AS users,
  COUNT_IF(converted)                               AS conversions,
  ROUND(100.0 * COUNT_IF(converted) / COUNT(*), 3)  AS conversion_rate_pct,
  SUM(total_ads)                                    AS impressions
FROM marketing_analytics.silver.ab_test_clean
GROUP BY 1, 2;

SELECT * FROM marketing_analytics.gold.ab_test_by_frequency
ORDER BY ad_bucket, test_group;

In [0]:
CREATE OR REPLACE TABLE marketing_analytics.gold.ab_test_by_day AS
SELECT
  CASE most_ads_day
    WHEN 'Monday'    THEN 1 WHEN 'Tuesday'  THEN 2 WHEN 'Wednesday' THEN 3
    WHEN 'Thursday'  THEN 4 WHEN 'Friday'   THEN 5 WHEN 'Saturday'  THEN 6
    WHEN 'Sunday'    THEN 7
  END                                               AS day_num,
  most_ads_day,
  test_group,
  COUNT(*)                                          AS users,
  COUNT_IF(converted)                               AS conversions,
  ROUND(100.0 * COUNT_IF(converted) / COUNT(*), 3)  AS conversion_rate_pct
FROM marketing_analytics.silver.ab_test_clean
GROUP BY 1, 2, 3;

SELECT * FROM marketing_analytics.gold.ab_test_by_day
ORDER BY day_num, test_group;

In [0]:
CREATE OR REPLACE TABLE marketing_analytics.gold.ab_test_by_daypart AS
SELECT
  CASE
    WHEN most_ads_hour BETWEEN 0  AND 5  THEN '1) Overnight 0-5'
    WHEN most_ads_hour BETWEEN 6  AND 11 THEN '2) Morning 6-11'
    WHEN most_ads_hour BETWEEN 12 AND 17 THEN '3) Afternoon 12-17'
    ELSE                                      '4) Evening 18-23'
  END                                               AS daypart,
  test_group,
  COUNT(*)                                          AS users,
  COUNT_IF(converted)                               AS conversions,
  ROUND(100.0 * COUNT_IF(converted) / COUNT(*), 3)  AS conversion_rate_pct
FROM marketing_analytics.silver.ab_test_clean
GROUP BY 1, 2;

SELECT * FROM marketing_analytics.gold.ab_test_by_daypart
ORDER BY daypart, test_group;

In [0]:
-- 1) Profit by ad-count bucket (feeds tile 3)
CREATE OR REPLACE TABLE marketing_analytics.gold.ab_test_bucket_profit AS
WITH p AS (
  SELECT
    ad_bucket,
    MAX(CASE WHEN test_group = 'ad'  THEN users       END) AS ad_users,
    MAX(CASE WHEN test_group = 'ad'  THEN conversions END) AS ad_conversions,
    MAX(CASE WHEN test_group = 'ad'  THEN impressions END) AS ad_impressions,
    MAX(CASE WHEN test_group = 'psa' THEN users       END) AS psa_users,
    MAX(CASE WHEN test_group = 'psa' THEN conversions END) AS psa_conversions
  FROM marketing_analytics.gold.ab_test_by_frequency
  GROUP BY ad_bucket
)
SELECT
  ad_bucket,
  ad_users,
  psa_users,
  ROUND(100.0 * (ad_conversions / ad_users - psa_conversions / psa_users), 3)                    AS lift_pts,
  ROUND((ad_conversions / ad_users - psa_conversions / psa_users) * ad_users, 1)                 AS extra_conversions,
  ROUND((ad_conversions / ad_users - psa_conversions / psa_users) * ad_users * 20, 0)            AS extra_revenue,
  ROUND(ad_impressions * 0.01, 0)                                                                AS ad_cost,
  ROUND((ad_conversions / ad_users - psa_conversions / psa_users) * ad_users * 20
        - ad_impressions * 0.01, 0)                                                              AS net_profit
FROM p;

-- 2) Lift by day of week (feeds tile 4)
CREATE OR REPLACE TABLE marketing_analytics.gold.ab_test_day_lift AS
SELECT
  day_num,
  most_ads_day,
  MAX(CASE WHEN test_group = 'ad'  THEN conversion_rate_pct END) AS ad_rate_pct,
  MAX(CASE WHEN test_group = 'psa' THEN conversion_rate_pct END) AS psa_rate_pct,
  ROUND(MAX(CASE WHEN test_group = 'ad'  THEN conversion_rate_pct END)
      - MAX(CASE WHEN test_group = 'psa' THEN conversion_rate_pct END), 3) AS lift_pts
FROM marketing_analytics.gold.ab_test_by_day
GROUP BY day_num, most_ads_day;

-- 3) One-row headline summary (feeds tiles 1 and 5)
-- Assumptions from the brief: $20 per conversion, $0.01 per impression. Change them here if the inputs change.
CREATE OR REPLACE TABLE marketing_analytics.gold.ab_test_roi_summary AS
WITH g AS (
  SELECT
    MAX(CASE WHEN test_group = 'ad'  THEN users       END) AS ad_users,
    MAX(CASE WHEN test_group = 'ad'  THEN conversions END) AS ad_conversions,
    MAX(CASE WHEN test_group = 'psa' THEN users       END) AS psa_users,
    MAX(CASE WHEN test_group = 'psa' THEN conversions END) AS psa_conversions
  FROM marketing_analytics.gold.ab_test_results
),
i AS (
  SELECT SUM(total_ads) AS ad_impressions
  FROM marketing_analytics.silver.ab_test_clean
  WHERE test_group = 'ad'
)
SELECT
  g.ad_users,
  g.psa_users,
  ROUND(100.0 * g.ad_conversions / g.ad_users, 3)                                                   AS ad_rate_pct,
  ROUND(100.0 * g.psa_conversions / g.psa_users, 3)                                                 AS psa_rate_pct,
  ROUND(100.0 * (g.ad_conversions / g.ad_users - g.psa_conversions / g.psa_users), 3)               AS lift_pts,
  ROUND(100.0 * ((g.ad_conversions / g.ad_users) / (g.psa_conversions / g.psa_users) - 1), 1)       AS relative_lift_pct,
  ROUND((g.ad_conversions / g.ad_users - g.psa_conversions / g.psa_users) * g.ad_users, 0)          AS extra_conversions,
  ROUND((g.ad_conversions / g.ad_users - g.psa_conversions / g.psa_users) * g.ad_users * 20, 0)     AS revenue,
  ROUND(i.ad_impressions * 0.01, 0)                                                                 AS cost,
  ROUND((g.ad_conversions / g.ad_users - g.psa_conversions / g.psa_users) * g.ad_users * 20
        - i.ad_impressions * 0.01, 0)                                                               AS net_profit,
  ROUND(i.ad_impressions * 0.01
        / ((g.ad_conversions / g.ad_users - g.psa_conversions / g.psa_users) * g.ad_users), 2)      AS breakeven_value_per_conversion
FROM g CROSS JOIN i;

In [0]:
SELECT * FROM marketing_analytics.gold.ab_test_roi_summary;

In [0]:
SELECT * FROM marketing_analytics.gold.ab_test_bucket_profit ORDER BY ad_bucket;

In [0]:
SELECT * FROM marketing_analytics.gold.ab_test_day_lift ORDER BY day_num;